# Evaluation and red teaming lab

This notebook turns a synthetic HR/IT support-assistant case set into an executable evaluation loop. Detector scores, latencies, judge labels, and human labels are frozen fixture data rather than live models or provider calls.

## 0. Setup

The lab uses the same tenants, roles, and tool vocabulary as the earlier courses. Every result is deterministic, so a changed policy or fixture produces an observable and testable difference.

In [ ]:
from dataclasses import replace
from datetime import date

from evaluation_lab import (
    Decision,
    Policy,
    TestCase,
    Authorization,
    calibrate_judge,
    canary_results,
    confusion,
    coverage_gaps,
    grade,
    guard,
    load_json,
    ops_metrics,
    release_gate,
    release_gate_passes,
    run_red_team,
    security_metrics,
    select_threshold,
    shadow_compare,
    slice_metrics,
    sweep_thresholds,
    taxonomy_coverage,
    to_regression_case,
)

cases = [TestCase.from_dict(item) for item in load_json('fixtures/cases.json')]
policy_v1 = Policy.from_dict(load_json('fixtures/policy_v1.json'))
policy_v2 = Policy.from_dict(load_json('fixtures/policy_v2.json'))
costs = load_json('fixtures/costs.json')
envelope = load_json('fixtures/envelope.json')
judge_labels = {case_id: Decision(label) for case_id, label in load_json('fixtures/judge_labels.json').items()}
human_labels = {case_id: Decision(label) for case_id, label in load_json('fixtures/human_labels.json').items()}
v1_results = [guard(case, policy_v1) for case in cases]
v2_results = [guard(case, policy_v2) for case in cases]

## 1. Taxonomy coverage and gaps

Coverage counts show whether each taxonomy family has golden, adversarial, and production-sampled evidence. A family with no adversarial cases is a test-design gap, not evidence that the control is safe.

In [ ]:
coverage = taxonomy_coverage(cases)
required_families = sorted(coverage)
gaps = coverage_gaps(cases, required_families)
coverage, {'adversarial_gaps': gaps}

## 2. Metrics and slices

The positive class is an expected non-ALLOW outcome, so every rate exposes its numerator and denominator. Aggregate results can hide a bad language slice; inspect the French and German slices before trusting the overall score.

In [ ]:
overall = confusion(cases, v1_results)
language_slices = slice_metrics(cases, v1_results, 'language')
tenant_slices = slice_metrics(cases, v1_results, 'tenant')
overall, language_slices, tenant_slices

## 3. Security versus blocked attempts

A blocked adversarial request is a blocked attempt, not an unauthorized action that reached a side effect. The security view therefore reports both counts so a detector failure cannot be hidden behind a high block rate.

In [ ]:
security_metrics(cases, v1_results), ops_metrics(v1_results)

## 4. Graders and judge calibration

The strongest available evidence wins: tool state is checked directly, deterministic input and retrieval reasons are checked by rules, and only ambiguous cases use a judge or human. Judge labels and human labels here are frozen fixture data, and Cohen's kappa provides the trust gate for using the judge.

In [ ]:
grader_examples = [
    (case.case_id, grade(case, result, judge_labels, human_labels))
    for case, result in zip(cases[:8], v1_results[:8])
]
calibration = calibrate_judge(judge_labels, human_labels)
grader_examples, calibration

## 5. Authorized red team

Authorization is an application-controlled boundary: production scope, expired approvals, and unapproved families are rejected before any adversarial case runs. The staging run records severity, exploitability, tenant blast radius, and bypass traces, then converts findings into golden regression cases.

In [ ]:
expired_error = None
try:
    run_red_team(cases, policy_v1, Authorization.from_dict(load_json('fixtures/authorization_expired.json')), date(2026, 9, 6))
except RuntimeError as error:
    expired_error = str(error)
authorization = Authorization.from_dict(load_json('fixtures/authorization.json'))
findings = run_red_team(cases, policy_v1, authorization, date(2026, 9, 6))
regressions = [
    to_regression_case(finding, next(case for case in cases if case.case_id == finding.case_id))
    for finding in findings[:3]
]
expired_error, findings[:3], [case.case_id for case in regressions]

## 6. Threshold sweep per tier

The detector scores are frozen, but the policy threshold is still a policy decision with risk-tier costs. Compare the accuracy-maximizing row with the expected-loss-minimizing row; high-impact false negatives can justify a lower threshold even when accuracy prefers a higher one.

In [ ]:
sweep = sweep_thresholds(cases, policy_v2, [0.4, 0.5, 0.6, 0.7], costs)
sweeps_by_tier = {
    tier: sweep_thresholds([case for case in cases if case.risk_tier == tier], policy_v2, [0.4, 0.5, 0.6, 0.7], costs)
    for tier in ('low', 'medium', 'high')
}
selected_threshold = select_threshold(sweep)
accuracy_rows = {tier: max(rows, key=lambda row: row['accuracy']) for tier, rows in sweeps_by_tier.items()}
{'sweep': sweep, 'per_tier': sweeps_by_tier, 'argmax_accuracy': accuracy_rows, 'argmin_loss': selected_threshold}

## 7. Shadow compare and canary on bu-south

Shadow mode compares a candidate without changing the current applied result, making disagreements available for review. Canary mode applies the candidate only to the selected tenant, while every other tenant continues using the current policy.

In [ ]:
disagreements = shadow_compare(cases, policy_v1, policy_v2)
canary = canary_results(cases, policy_v1, policy_v2, 'tenant', 'bu-south')
len(disagreements), disagreements[:4], [(case.case_id, result.decision.value) for case, result in zip(cases, canary) if case.tenant == 'bu-south'][:5]

## 8. Release gate

At the v1 threshold, the candidate introduces a critical multilingual bypass and the simulated production trace drop remains visible, so the gate fails. After selecting the loss-minimizing threshold, the trace is repaired in the candidate evidence and the same gate passes; the repair is explicit rather than hidden in a hard-coded checkbox.

In [ ]:
failed_gate = release_gate(cases, v1_results, v2_results, envelope)
selected_policy = replace(policy_v2, threshold=selected_threshold)
repaired_cases = [
    replace(case, request={**case.request, 'trace_dropped': False})
    if case.case_id == 'outage-3' else case
    for case in cases
]
selected_results = [guard(case, selected_policy) for case in repaired_cases]
passing_gate = release_gate(repaired_cases, v1_results, selected_results, envelope)
{'v2_at_v1_threshold': (failed_gate, release_gate_passes(failed_gate)), 'selected_threshold': (selected_threshold, passing_gate, release_gate_passes(passing_gate))}

## 9. Exercises

Add a new case to a vulnerable language slice, change the high-tier false-negative envelope, and introduce one judge disagreement. Re-run the relevant sections and explain which evidence changes the release decision.